# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library, following the Croissant schema.

### Dataset Source
The dataset schema is published at:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

**Note:** All entities (record sets, fields, columns) are referenced by their `@id` throughout the notebook.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the Croissant dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and display main metadata attributes (using methods/properties, not dict access)
print("\033[1mDataset Title:\033[0m", dataset.metadata.name)
print("\033[1mDataset Description:\033[0m", dataset.metadata.description)
print("\033[1mIdentifier:\033[0m", dataset.metadata.identifier)
print("\033[1mVersion:\033[0m", dataset.metadata.version)
print("\033[1mDate Published:\033[0m", dataset.metadata.datePublished)
print("\033[1mLicense:\033[0m", dataset.metadata.license)
print("\033[1mFields containing personal sensitive information: \033[0m", getattr(dataset.metadata, 'personalSensitiveInformation', None))

## 2. Data Overview
Let's review available record sets in the dataset and explore their `@id` values, along with fields inside them. This is useful for referencing fields precisely in later analysis steps.

For each RecordSet, we'll list its `@id` and all constituent Fields and Columns (referenced by their `@id`).

In [ ]:
# Discover Record Sets -- their @id and field/column structure
print("\033[1mListing dataset record sets and their fields\033[0m\n")

record_set_ids = []
for recset in dataset.metadata.recordSets:
    print(f"RecordSet name: {recset.name}\n  @id: {recset.id}")
    record_set_ids.append(recset.id)
    print("  Fields:")
    for field in recset.fields:
        # Each field may have a column associated
        print(f"    - {field.name} (@id: {field.id})")
        for col in getattr(field, 'columns', []):
            print(f"          column @id: {col.id}")
    print()
if not record_set_ids:
    print("Warning: No record sets defined in metadata!\nIf so, check dataset documentation or inspect schema distribution for table resources.")

Let's inspect and print a sample of records from each available record set. Use the `@id` of the record set with the loader.

In [ ]:
# For demo: Print a sample record from each record set by @id
for record_set_id in record_set_ids:
    print(f"\033[1mSample record from RecordSet @id: {record_set_id}\033[0m")
    sample_iter = dataset.records(record_set=record_set_id)
    try:
        sample = next(sample_iter)
        pprint.pprint(sample)
    except StopIteration:
        print("  No records found for this record set.")
    print()

## 3. Data Extraction
Load data from a chosen record set into a DataFrame. This allows flexible analysis and processing using Pandas. Always use the precise record set and field/column `@id` values identified above.

In [ ]:
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded RecordSet @id: {record_set_id}  (shape: {dataframes[record_set_id].shape})")
        if not dataframes[record_set_id].empty:
            print("Fields:", dataframes[record_set_id].columns.tolist())
            display(dataframes[record_set_id].head(3))
        else:
            print("No records loaded for this set.")
    except Exception as e:
        print(f"Error loading record set {record_set_id}: {e}")
        dataframes[record_set_id] = pd.DataFrame()

Identify a tabular RecordSet and select numeric/categorical fields for exploratory analysis below.

We'll continue using the first non-empty record set for analysis. (Modify the cell to use any specific RecordSet or field `@id` as needed.)

In [ ]:
# Pick first non-empty DataFrame for demonstration:

selected_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        selected_record_set_id = rsid
        break

if selected_record_set_id is None:
    raise RuntimeError("No records available in any record set!")

print(f"Selected RecordSet for analysis: {selected_record_set_id}")
df = dataframes[selected_record_set_id]
print("Available columns:")
print(df.columns.tolist())

## 4. Exploratory Data Analysis (EDA)
Let's perform basic processing and summary statistics:
- Filter on a numeric field (e.g., age, diagnosis interval, etc.)
- Normalize numeric values
- Group by a clinically meaningful categorical field

**NOTE:** Replace `<numeric_field_id>` and `<group_field_id>` below with the actual `@id` of the columns you want. (Below, we'll pick commonly encountered fields in clinical cancer datasets. Inspect the columns printed above for available choices.)

In [ ]:
# Example: Identify a numeric field and group field by inspecting column names
# You may customize these based on your dataset fields

numeric_field_id = None
group_field_id = None

# Try to auto-select likely numeric and group fields (override as needed) --
for col in df.columns:
    # Pick a numeric field containing 'age' or 'interval', else any float/int dtype
    if numeric_field_id is None and (('age' in col.lower()) or ('interval' in col.lower())):
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
    # For grouping, pick anatomical location or sex if available
    if group_field_id is None and (('location' in col.lower()) or ('sex' in col.lower()) or ('msi' in col.lower())):
        group_field_id = col

# fallback: pick first numeric and first object field if auto not found
if numeric_field_id is None:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

if group_field_id is None:
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]):
            group_field_id = col
            break

print(f"Numeric field selected: {numeric_field_id}")
print(f"Grouping field selected: {group_field_id}")

# Proceed only if numeric field exists
if numeric_field_id is None:
    raise Exception("No numeric field found. Please set 'numeric_field_id' manually.")

# Remove outliers: filter on numeric value (here arbitrarily > 10)
threshold_value = 10
filtered_df = df[df[numeric_field_id] > threshold_value]
print(f"\nFiltered records where {numeric_field_id} > {threshold_value} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalization (z-score) of the selected numeric variable
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized column '{numeric_field_id}' for filtered records:")
display(filtered_df[[numeric_field_id, norm_col]].head())

# Grouped summary
if group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
    print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
    display(grouped.head())
else:
    print(f"Grouping field '{group_field_id}' not found for this table.")

## 5. Visualization
Visualize the distribution of the selected numeric field, and (if available) compare distributions grouped by a clinical or molecular feature.

Below is an example using Matplotlib and Seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7,4))
sns.histplot(filtered_df[numeric_field_id], kde=True, bins=12, color="dodgerblue")
plt.title(f"Distribution of {numeric_field_id} (> {threshold_value})")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

if group_field_id in filtered_df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion

This notebook demonstrated how to:
- Load Croissant schema-based datasets with `mlcroissant`
- Explore dataset metadata and tabular record sets, referencing all entities by their `@id`
- Extract record sets and perform exploratory data analysis, including filtering, normalization, and grouping
- Visualize numeric and categorical data in the clinical dataset context

For advanced analysis, continue referencing fields and tables by their `@id`, and always inspect the Croissant metadata for definitions and provenance. Consult the [mlcroissant documentation](https://mlcommons.github.io/croissant/api.html) for further capabilities and multi-dataset support.